In [1]:
import pandas as pd
import os 
import json
import shutil

In [2]:
train_path = 'desegma-it.subTaskA.shared.train.0923-1220 (1).csv'
test_path = 'desegma-it.subTaskA.with_labels.test.1117_1835.csv'
out_dir = 'profiling_input' # cartella contenente i text
document_info_path = 'document_info.json' # documento contenente il dizionario
                                          # che associa il text a label e allo split

In [3]:
def create_balanced_sets(df_tr, df_ts):
    train_subset = pd.concat([
        df_tr[df_tr['label']==0].sample(1000, random_state=26), #random_state fissato a 26 per generare ogni volta gli stessi subsets
        df_tr[df_tr['label']==1].sample(1000, random_state=26)
    ])
    # usando il metodo drop, salviamo in una nuova variabile solo
    # i dati che ancora non sono stati usati per il subset di tr
    remaining_train = df_tr.drop(train_subset.index) 
    
    val_subset = pd.concat([
        remaining_train[remaining_train['label']==0].sample(500, random_state=26),
        remaining_train[remaining_train['label']==1].sample(500, random_state=26)
    ])
    
    test_subset = pd.concat([
        df_ts[df_ts['label']==0].sample(500, random_state=26),
        df_ts[df_ts['label']==1].sample(500, random_state=26)
    ])
    
    return train_subset, val_subset, test_subset
    

In [4]:
df_tr = pd.read_csv(train_path)
df_ts = pd.read_csv(test_path)
train_set, val_set, test_set = create_balanced_sets(df_tr, df_ts)

In [5]:
# identifica lo split di appartenenenza per popolare il dict JSON
# e creare ID univoci per doc_id presenti nei subsets
def get_split(file_path):
    if 'train' in file_path:
        return 'training'
    elif 'test' in file_path:
        return 'test'
    else:
        return 'validation'

In [6]:
user_info_dict = {} 
# lista di tuple per processare l'assegnazione dello split in un unico
# ciclo for 
subsets = [
    (train_set, train_path),
    (val_set, 'validation'), # quando il percorso contiene la stringa 'validation'
    (test_set, test_path)
]

for df_subset, file_path in subsets:
    split_type = get_split(file_path)
    
    for idx, row in df_subset.iterrows(): # .iterrows() permette di scorrere sulle righe
        doc_id=f"{split_type}_{idx}" # ogni doc_id corrisponde ad una row del subset 
        user_info_dict[doc_id]={
            'label':int(row['label']),
            'split': split_type
        }
        
        out_file_name = f"{doc_id}.txt"
        out_file_path = os.path.join(out_dir, out_file_name)
        
        # eliminazione di eventuali spazi bianchi
        cleaned_text = str(row['text']).strip()
        
        # contenuto di ogni doc_id
        with open(out_file_path, 'w', encoding='utf-8') as out_f:
            out_f.write(cleaned_text)

In [7]:
with open(document_info_path, 'w', encoding='utf-8') as json_f:
    json.dump(user_info_dict, json_f, indent=4) # .dump() trasforma il dizionario in formato JSON nel file json_f (document_info_path)

In [8]:
# creazione cartella zip per il tool Profiling-UD 
zip_folder = 'profiling_input_zip'
folder_selected = 'profiling_input'
shutil.make_archive(zip_folder, 'zip', folder_selected)

'c:\\Users\\Alessia\\Desktop\\lc2-project\\profiling_input_zip.zip'